# Intoduction to Langchain by building email translator
There are three important sections in LangChain
* Model - Base LLM which is corresponsible to generating output
* Prompt - A helper function which provides input to the LLM in more sensible way 
* Parser - Transforms the output generated into useful manner

# Example 1 (model usage): Email translation

In [1]:
from langchain_ollama.chat_models import ChatOllama

llm = ChatOllama(
    model='mistral',
    temperature=0.75
)

In [2]:
# Actual content to be translated
email = "Hi John, Don't you understand the importance of it? this is very important for production. Fix it at any cost"

In [3]:
# Requirement
style = "korean language with calm & respectful tone"

In [4]:
# Input building
prompt = f"Translate the text which is delimited by three backticks " \
f"with the style as {style} " \
f"text : ```{email}``` "
msg = [ {'role' : 'user', 'content' : prompt} ]

In [5]:
# Generation of output

response = llm.invoke(msg)

In [6]:
print(response.content)

안녕하세요, 존, 이것의 중요성에 대해서 신경쓰지 않으시나요? 제작 단계에서는 매우 중요합니다. 어떤 비용을 들여도 고쳐주세요.

(Translation: Hello, John, are you not aware of its importance? It is very important for production. Fix it at any cost.)


# Example 2 (Prompt usage)
* As we see in the last example, we need to provide more information along with our actual input `email`
* To handle this, `prompts` abstraction layer is provided.
* Lets rewrite with prompts

In [7]:
from langchain.prompts import ChatPromptTemplate

In [8]:
template = """ You are an helpful translating assistant. You are able to translate between any language.
Please translate the text delimited with three backticks into style matching with {style} 
text : ```{text}``` """

In [9]:
prompt_template = ChatPromptTemplate.from_template(
    template=template
)

In [10]:
# Actual content to be translated
email = "Hi John, Don't you understand the importance of it? this is very important for production. Fix it at any cost"

# Requirement
style = "korean language with calm & respectful tone"

In [11]:
formated_input  = prompt_template.format_messages(
    style = style,
    text = email
)
formated_input

[HumanMessage(content=" You are an helpful translating assistant. You are able to translate between any language.\nPlease translate the text delimited with three backticks into style matching with korean language with calm & respectful tone \ntext : ```Hi John, Don't you understand the importance of it? this is very important for production. Fix it at any cost``` ", additional_kwargs={}, response_metadata={})]

In [12]:
# Generate output
response = llm.invoke(formated_input)

print(response.content)

안녕하세요, 존, 이것의 중요성에 어떻게 이해하지 않는 건가요? 생산 위험에 있기 때문에 일찍인척하도록 노력하시면 됩니다. 모든 비용을 포기하고 수정해야 합니다.

(Translated: Hello, John, don't you understand the importance of it? This is very important for production. Fix it at any cost)


# Example 3 (Output Parser usage)
* As we see in the last sample, the generated output contains both korean as well as english
* To control the output & format in required manner, `output parsers` are introduced
* This has wide range of applications
  * will help in organizing between the agents
  * Provide function calling
* Lets rewrite the program

In [13]:
# Expected output:
{
    "input_language" : "English",
    "output_language" : "korean",
    "output" : "<generated output>" 
}

{'input_language': 'English',
 'output_language': 'korean',
 'output': '<generated output>'}

In [14]:
transaltion_parser = """ You are a helpful assistant to extract information from query

Query : Translate the  text delimited with three backticks into style matching with {style} 
text : ```{text}```

Your task is to extract the following information
input_language : language of input text
output_language : expected output language
output_text : translated value

The output will be generated in JSON format with following keys:
input_language
output_language
output_text
 """

In [15]:
prompt_template = ChatPromptTemplate.from_template(
    template=transaltion_parser
)

In [16]:
# Actual content to be translated
email = "Hi John, Don't you understand the importance of it? this is very important for production. Fix it at any cost"

# Requirement
style = "korean language with calm & respectful tone"

In [17]:
formated_input  = prompt_template.format_messages(
    style = style,
    text = email
)
formated_input

[HumanMessage(content=" You are a helpful assistant to extract information from query\n\nQuery : Translate the  text delimited with three backticks into style matching with korean language with calm & respectful tone \ntext : ```Hi John, Don't you understand the importance of it? this is very important for production. Fix it at any cost```\n\nYour task is to extract the following information\ninput_language : language of input text\noutput_language : expected output language\noutput_text : translated value\n\nThe output will be generated in JSON format with following keys:\ninput_language\noutput_language\noutput_text\n ", additional_kwargs={}, response_metadata={})]

In [18]:
# Generate output
response = llm.invoke(formated_input)

print(response.content)

 {
      "input_language": "English",
      "output_language": "Korean",
      "output_text": "안녕하세요 존, 그것의 중요성을 이해하지 못하시니까요? 생산에 매우 중요합니다. 어떻게든 수정하세요."
   }
